**`validate_occupancy`**

Scores curated `occupancy_type` against hand-labeled ground-truth points
and gates on per-class F1, so a change that improves one class by
wrecking another fails loudly.

Aggregate agreement is deliberately not the gate. It sat near 65% while
Single-Family accuracy was 37.6%, so a single scalar looked adequate
while the largest error mode in the dataset went unnoticed. Recall alone
is not the gate either: it would fail a change that trades a little
recall for more precision, and pass one that inflates recall by
labelling everything a single class.

# Configure

In [ ]:
import argparse
import sys
from pathlib import Path

import pandas as pd

# CHEER-specific configuration (counties, survey path, band collapse)
# lives in cheer_linkage.py beside this notebook; the linkage and scoring
# themselves are generic and live in openplaces.io.curator.validation.
# Resolve the repository root from any working directory (Jupyter runs
# notebooks from their own folder; scripts run from the repository root),
# so the import below works either way. It has to happen here rather than
# beside its first use, because the parser below defaults to a path this
# module defines.
root = Path.cwd()
while not (root / 'src' / 'openplaces').exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / 'notebooks' / '05_curate'))
from openplaces.io.delivery import delivery_accuracy_dir  # noqa: E402
from cheer_linkage import (  # noqa: E402
    BASELINE_PATH,
    COUNTIES,
    check_baseline_coverage,
    link_ground_truth,
    score_sources,
)

In [ ]:
parser = argparse.ArgumentParser(description='Validate curated occupancy.')
parser.add_argument('--recipe_id', default='US_footprint-cheer-2026')
parser.add_argument('--admin_ids', nargs='*', default=None)
# Default None resolves to cheer_linkage.BASELINE_PATH below: the baseline
# is survey-derived (third-party data), so it lives in the cache tree,
# never beside the recipe in the repository.
parser.add_argument('--baseline', default=None)
parser.add_argument('--write_baseline', action='store_true')
parser.add_argument('--tolerance', type=float, default=0.01)
# Scored tables go into the delivery's own `accuracies/` folder: how well
# the inventory scores travels with the inventory. Only the row-level
# linkage stays in the cache -- it carries survey addresses (see
# link_ground_truth's `save`), and the bundle is shared.
parser.add_argument(
    '--out_dir', default=str(delivery_accuracy_dir('US_footprint-cheer-2026'))
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--recipe_id US_footprint-cheer-2026 --verbose '

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display parsed arguments
args

# Validate occupancy against ground truth

In [ ]:
counties = tuple(args.admin_ids) if args.admin_ids else COUNTIES
linked = link_ground_truth(counties, verbose=args.verbose)
linked.shape

In [ ]:
# Score the vote and every input it arbitrates. A vote that scores worse
# than one of its own inputs on a class is discarding evidence, which a
# single-column score cannot show.
table = score_sources(linked)

out_dir = Path(args.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
scores_path = out_dir / f'{args.recipe_id}_occupancy-scores.csv'
table.to_csv(scores_path, index=False)
print(f'wrote scores to {scores_path}')
table

In [ ]:
# Gate on F1 against the cache-tree baseline (or an explicit --baseline).
# Merge on source *and* class: the baseline holds one block per evidence
# source, so joining on class alone would compare the vote against every
# source's row at once.
baseline_path = args.baseline or BASELINE_PATH
if args.write_baseline:
    table.to_csv(baseline_path, index=False)
    print(f'Baseline written: {baseline_path}')
else:
    baseline = pd.read_csv(baseline_path)
    # A source missing from `table` would drop its baseline rows
    # from the merge silently, letting the gate pass on fewer
    # sources than it reports. Fail instead.
    check_baseline_coverage(table, baseline)
    merged = table.merge(baseline, on=['source', 'class'], suffixes=('', '_base'))
    merged['d_f1'] = merged['f1'] - merged['f1_base']
    comparison = merged[['source', 'class', 'f1_base', 'f1', 'd_f1']]
    print(comparison.to_string(index=False))
    # Written before the gate raises, so a failing run leaves behind the
    # table that explains which class moved and by how much.
    comparison_path = out_dir / f'{args.recipe_id}_occupancy-scores-vs-baseline.csv'
    comparison.to_csv(comparison_path, index=False)
    print()
    print(f'wrote comparison to {comparison_path}')
    # Only the vote is gated: the inputs are reported for diagnosis, and a
    # deliberate change to how one is read should not fail the run.
    regressed = merged[
        merged['source'].eq('final_vote')
        & merged['class'].ne('ALL')
        & (merged['d_f1'] < -args.tolerance)
    ]
    if len(regressed):
        raise SystemExit(f'FAIL: {len(regressed)} class(es) lost F1')

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

try:
    convert_to_script(commit=True)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')